In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image settings
IMG_SIZE = (128, 128)
BATCH_SIZE = 4

# Data Generator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training Dataset
train_data = train_datagen.flow_from_directory(
    "fashion_dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

# Validation Dataset
validation_data = train_datagen.flow_from_directory(
    "fashion_dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=True
)

print("\nClass Labels:")
print(train_data.class_indices)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# CNN Model
model = Sequential([

    Conv2D(
        filters=32,
        kernel_size=(3,3),
        activation="relu",
        input_shape=(128,128,3)
    ),

    MaxPooling2D(pool_size=(2,2)),

    Flatten(),

    Dense(128, activation="relu"),

    Dense(3, activation="softmax")
])

# Compile Model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Model Summary
model.summary()

In [ ]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=5
)

print("\nTraining and Validation Accuracy:\n")

for i in range(5):
    print(
        f"Epoch {i+1} -> "
        f"Train Accuracy: {history.history['accuracy'][i]:.4f} | "
        f"Validation Accuracy: {history.history['val_accuracy'][i]:.4f}"
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Test Dataset
test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "fashion_dataset/test",
    target_size=(128,128),
    batch_size=1,
    class_mode="categorical",
    shuffle=False
)

# Predict
predictions = model.predict(test_data)

predicted_labels = np.argmax(predictions, axis=1)

true_labels = test_data.classes

class_names = list(test_data.class_indices.keys())

# Calculate Accuracy
accuracy = np.mean(predicted_labels == true_labels)

print(f"\nTest Accuracy: {accuracy*100:.2f}%")

print("\nMisclassified Images:\n")

misclassified = False

for i in range(len(true_labels)):

    if predicted_labels[i] != true_labels[i]:

        misclassified = True

        image = test_data[i][0][0]

        plt.figure(figsize=(3,3))
        plt.imshow(image)
        plt.axis("off")

        plt.title(
            f"Actual: {class_names[true_labels[i]]}\n"
            f"Predicted: {class_names[predicted_labels[i]]}"
        )

        plt.show()

if not misclassified:
    print("No misclassified images. All test images were classified correctly.")